# Preprocessing

Run from the repo root.

In [ ]:
import pandas as pd
import similarity.preprocessing as pp

raw = pp.load_raw()
raw.shape

In [ ]:
raw.head(3)

## brand

In [ ]:
brand = raw["brand"].head(10)
pd.DataFrame({"raw": brand, "clean": brand.map(pp.parse_brand)})

In [ ]:
pp.parse_brand("Generic"), pp.parse_brand("N/A")

## colour

In [ ]:
colour = raw["colour"].dropna().head(10)
pd.DataFrame({"raw": colour, "clean": colour.map(pp.parse_colours)})

## weight

In [ ]:
pattern = raw["weight"].str.replace(r"[\d.]+", "N", regex=True)
examples = raw.groupby(pattern)["weight"].first()
pd.DataFrame({"raw": examples, "grams": examples.map(pp.parse_weight_grams)})

In [ ]:
pp.parse_weight_grams("113 Kilograms"), pp.parse_weight_grams("4.54 g")

In [ ]:
raw["weight"].map(pp.parse_weight_grams).describe()

## price and rating

In [ ]:
raw["sales_price"].map(pp.parse_price).describe()

In [ ]:
raw["rating"].map(pp.parse_rating).describe()

## reviews

In [ ]:
reviews = pd.Series(
    [pp.parse_review_count(d, c) for d, c in zip(raw["product_details__k_v_pairs"], raw["no__of_reviews"])]
)
reviews.describe()

In [ ]:
reviews.isna().sum(), (reviews == 0).sum()

## category, browsenode, image

In [ ]:
raw["sales_rank_in_child_category"].head(5).map(pp.parse_category)

In [ ]:
raw["browsenode"].head(5).map(pp.parse_browsenode)

In [ ]:
raw["medium"].head(3).map(pp.parse_first_url)

## build

In [ ]:
products = pp.build_products(raw)
products.shape

In [ ]:
products.head()

In [ ]:
products.dtypes

In [ ]:
pp.fill_summary(products)

In [ ]:
products["asin"].duplicated(keep=False).sum()

## rating_adjusted

In [ ]:
products.loc[products["review_count"] == 1, ["rating", "review_count", "rating_adjusted"]].head()

In [ ]:
products.sort_values("rating_adjusted", ascending=False)[["name", "rating", "review_count", "rating_adjusted"]].head(10)

## save

In [ ]:
pp.validate(products)

In [ ]:
pp.save(products)

In [ ]:
pp.load_products().shape